<a href="https://colab.research.google.com/github/elhamod/BA305_Fall_2026/blob/main/Session%2023%20-%20Optimization%20II%20-%20LP%20and%20Network%20Models/lab10_LP_Networks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Network Optimization Case: Vaccine Distribution
Gerry Tsoukalas (gerryt@bu.edu)

## Objectives:

In this module, we will learn how to use Linear Programming to solve a network optimization problem. We will:

*   translate a problem description into a mathematical (algebraic) model
*   embed this model into a python optimization framework
*   choose the correct solution algorithm
*   solve and interpret the model


## Problem Description

A company makes a vaccine in four plants and sells it through four regional distribution centers (DCs). Any plant can ship to any DC. **How many tons should each plant make and ship to each DC to maximize annual profit**, without exceeding any plant's capacity or any region's demand? The full case is on the slides; the numbers the model uses are below (all costs and prices in $ per ton).

| Plant (i) | Capacity (tons) | Production cost |
| --- | --- | --- |
| 1 Newark | 100 | 34,900 |
| 2 Los Angeles | 200 | 32,200 |
| 3 Rotterdam | 120 | 38,350 |
| 4 Kuala Lumpur | 250 | 23,400 |

| Market / DC (j) | Max demand (tons) | Selling price |
| --- | --- | --- |
| 1 Newark | 150 | 55,500 |
| 2 Sao Paulo | 75 | 61,100 |
| 3 Rotterdam | 200 | 57,800 |
| 4 Tokyo | 100 | 62,650 |

Transportation cost (freight and tariffs) from plant i to DC j:

| From \ To | Newark | Sao Paulo | Rotterdam | Tokyo |
| --- | --- | --- | --- | --- |
| Newark | 0 | 12,225 | 9,075 | 21,450 |
| Los Angeles | 4,500 | 16,500 | 13,350 | 17,850 |
| Rotterdam | 9,150 | 12,600 | 0 | 12,525 |
| Kuala Lumpur | 21,450 | 18,450 | 15,150 | 5,925 |

## Setup

Pyomo is the modeling package: it lets us write the model in Python. GLPK is the solver: the engine that actually finds the optimal plan. The `apt-get` line installs GLPK and works in **Google Colab**; it will not work on a Mac or on Windows.

In [ ]:
%%capture
# Install Pyomo and the GLPK solver, then load the packages.
!pip install pyomo
!apt-get install -y -qq glpk-utils
import pyomo.environ as pyo
import pandas as pd

## Optimization Model

### Decision variables

$x_{ij}$ is the number of tons made in plant $i$ and shipped to DC $j$. With 4 plants and 4 DCs that is 4 × 4 = 16 decision variables, for example `x13` = tons made in Newark and shipped to Rotterdam.

In [ ]:
# Create the model and one variable per plant-DC pair (in tons).
model = pyo.ConcreteModel()

model.x11 = pyo.Var(bounds=(0, None))
model.x12 = pyo.Var(bounds=(0, None))
model.x13 = pyo.Var(bounds=(0, None))
model.x14 = pyo.Var(bounds=(0, None))
model.x21 = pyo.Var(bounds=(0, None))
model.x22 = pyo.Var(bounds=(0, None))
model.x23 = pyo.Var(bounds=(0, None))
model.x24 = pyo.Var(bounds=(0, None))
model.x31 = pyo.Var(bounds=(0, None))
model.x32 = pyo.Var(bounds=(0, None))
model.x33 = pyo.Var(bounds=(0, None))
model.x34 = pyo.Var(bounds=(0, None))
model.x41 = pyo.Var(bounds=(0, None))
model.x42 = pyo.Var(bounds=(0, None))
model.x43 = pyo.Var(bounds=(0, None))
model.x44 = pyo.Var(bounds=(0, None))

# Shorter names, so the formulas below read like the slides.
x11 = model.x11
x12 = model.x12
x13 = model.x13
x14 = model.x14
x21 = model.x21
x22 = model.x22
x23 = model.x23
x24 = model.x24
x31 = model.x31
x32 = model.x32
x33 = model.x33
x34 = model.x34
x41 = model.x41
x42 = model.x42
x43 = model.x43
x44 = model.x44

### Objective function

Profit = revenue − production cost − transportation cost.

* **Revenue**: everything arriving at DC $j$ sells at DC $j$'s price, so each price multiplies the column of tons going *into* that DC (`x11 + x21 + x31 + x41` for Newark).
* **Production cost**: everything made at plant $i$ costs plant $i$'s unit cost, so each cost multiplies the row of tons going *out of* that plant (`x11 + x12 + x13 + x14` for Newark).
* **Transportation cost**: each route has its own cost per ton.

In [ ]:
# Annual profit to maximize, in dollars.
model.obj = pyo.Objective(expr =
      55500*(x11 + x21 + x31 + x41)      # revenue: Newark DC
    + 61100*(x12 + x22 + x32 + x42)      # revenue: Sao Paulo DC
    + 57800*(x13 + x23 + x33 + x43)      # revenue: Rotterdam DC
    + 62650*(x14 + x24 + x34 + x44)      # revenue: Tokyo DC
    - 34900*(x11 + x12 + x13 + x14)      # production: Newark plant
    - 32200*(x21 + x22 + x23 + x24)      # production: Los Angeles plant
    - 38350*(x31 + x32 + x33 + x34)      # production: Rotterdam plant
    - 23400*(x41 + x42 + x43 + x44)      # production: Kuala Lumpur plant
    - ( 0*x11 + 12225*x12 + 9075*x13 + 21450*x14     # transportation
       +4500*x21 + 16500*x22 + 13350*x23 + 17850*x24
       +9150*x31 + 12600*x32 + 0*x33 + 12525*x34
       +21450*x41 + 18450*x42 + 15150*x43 + 5925*x44)
, sense = pyo.maximize)

### Constraints

* **Production constraints** `C1`–`C4`: a plant cannot ship more than its capacity (one row of the shipping table).
* **Demand constraints** `D1`–`D4`: a DC cannot receive more than its region's demand (one column of the shipping table).

In [ ]:
# Production constraints: plant i ships at most its capacity.
model.C1 = pyo.Constraint(expr = x11 + x12 + x13 + x14 <= 100)   # Newark
model.C2 = pyo.Constraint(expr = x21 + x22 + x23 + x24 <= 200)   # Los Angeles
model.C3 = pyo.Constraint(expr = x31 + x32 + x33 + x34 <= 120)   # Rotterdam
model.C4 = pyo.Constraint(expr = x41 + x42 + x43 + x44 <= 250)   # Kuala Lumpur

# Demand constraints: DC j receives at most its region's demand.
model.D1 = pyo.Constraint(expr = x11 + x21 + x31 + x41 <= 150)   # Newark
model.D2 = pyo.Constraint(expr = x12 + x22 + x32 + x42 <=  75)   # Sao Paulo
model.D3 = pyo.Constraint(expr = x13 + x23 + x33 + x43 <= 200)   # Rotterdam
model.D4 = pyo.Constraint(expr = x14 + x24 + x34 + x44 <= 100)   # Tokyo

### Solve the model

As in Lab 9: GLPK solves the linear program, `opt.options['ranges']` writes the sensitivity report to `sens.txt`, and `symbolic_solver_labels=True` makes that report use our own names (`x11`, `C1`, `D4`, ...).

In [ ]:
# Solve with GLPK and save the sensitivity report to sens.txt.
opt = pyo.SolverFactory('glpk')
opt.options['ranges'] = 'sens.txt'
results = opt.solve(model, symbolic_solver_labels=True)
print(results.solver.termination_condition)
print("Optimal annual profit ($) =", model.obj())

### Results

Sixteen numbers are easier to read as a table: one row per plant (where the vaccine is made) and one column per DC (where it is sold).

In [ ]:
# Optimal shipments in tons: rows are plants, columns are DCs.
shipments = pd.DataFrame(
    [[x11(), x12(), x13(), x14()],
     [x21(), x22(), x23(), x24()],
     [x31(), x32(), x33(), x34()],
     [x41(), x42(), x43(), x44()]],
    index=['Newark', 'Los Angeles', 'Rotterdam', 'Kuala Lumpur'],
    columns=['Newark', 'Sao Paulo', 'Rotterdam', 'Tokyo'])
shipments

## Sensitivity Analysis

Read it the same way as in Lab 9: page 1 has one row per constraint (`C1`–`C4` are the plants, `D1`–`D4` the DCs), pages 2 and 3 one row per route. A positive **Marginal** (shadow price) is the extra profit from one more ton of that plant's capacity or that region's demand, valid within the **Activity range**.

In [ ]:
# Print the sensitivity report GLPK wrote during the solve.
print(open('sens.txt').read())

### Follow up Questions and Exercises

**Question 1.** Which plants run at full capacity, and which one has capacity to spare?

**Answer**

*Provide your answer here*

**Question 2.** What are the shadow prices of Rotterdam's production capacity (`C3`) and of Tokyo's demand (`D4`)? What does each one mean for the business?

**Answer**

*Provide your answer here*

**Question 3.** Why are the production shadow prices (`C1`–`C4`) lower than the demand shadow prices (`D1`–`D4`)?

**Answer**

*Provide your answer here*